# 21 — Labels: manual sheets, checks, automatic matcher and split (Milestone 3)

Owner: Member 2. Milestone 3 needs a label for every (BibTeX entry, arXiv reference) cell of a publication:
a labelled entry either matches **one** reference of the same publication or none. This notebook makes the
two kinds of labels the statement asks for and the split.

- **Manual**: each member labels 5 publications of their own ID range (20 for the group), every entry of
  each one, in a spreadsheet made here. The sheet shows 3 title hints, never the automatic label.
- **Automatic**: a cascade of rules (arXiv id, DOI, exact title, fuzzy title with author or year), applied to
  publications outside the manual pool (the statement asks for at least 10% of them). It labels every
  entry of a publication it is applied to.
- **Split**: test = 1 manual + 1 automatic publication, valid = the same, train = the rest (`split.json`).

| Section | Functions |
|---|---|
| 0. Setup | loads `20_m3_data` (helpers, `load_m3`, `MEMBER_RANGES`) and `22_features` (cleaning) |
| 1. Publications to label | `manual_candidates`, `pick_manual`, `write_selection`, `read_selection` |
| 2. Label sheets | `export_label_sheet`, `export_sheets` |
| 3. Reading and checking manual labels | `repair_arxiv_id`, `read_manual`, `validate_labels`, `print_validation` |
| 4. Automatic matcher | `AUTO_PARAMS`, `auto_match_pub`, `auto_label_all`, `auto_vs_manual`, `tune_auto_params` |
| 5. Split | `split_pool`, `pick_split`, `make_split` |
| 6. Labels for 22 | `build_labels`, `load_labels` |
| 7. Full run | `prepare_manual_sheets` (before labelling), `run_labeling` (after; called by `25_run_m3`) |
| 8. Tests | `LabelTests` |

**Files** (Drive `MyDrive/arxiv-project/`, **never committed**):

| File | Written by | Content |
|---|---|---|
| `labels/selection.csv` | `prepare_manual_sheets` | `member, arxiv_id, bib_source, n_entries, n_refs`: the 20 publications to label |
| `labels/todo/<member>/<fid>.csv`, `<fid>_refs.csv` | `prepare_manual_sheets` | the empty sheets to open in Google Sheets |
| `labels/manual/<fid>.csv` | each member | the filled sheet, downloaded as CSV |
| `labels/auto/auto_labels.parquet` | `run_labeling` | `arxiv_id, citation_key, ref_arxiv_id (null = no match), rule, score` |
| `labels/split.json` | `run_labeling` | `{"seed", "test": {"manual", "auto"}, "valid": {"manual", "auto"}}` |
| `labels/labels.parquet` | `run_labeling` | manual and automatic labels together, read by `22` through `load_labels()` |
| `logs/m3_labels.json` | `run_labeling` | label counts, automatic matcher parameters, precision and recall against the manual labels |

## 0. Setup

`20_m3_data` gives `H12`, `H03`, `load_m3`, `M3_DIR`, `LABEL_DIR`, `MEMBER_RANGES`, `member_of`,
`M3_PARSER_VERSION`, `MANUAL_POOL` and `SPLIT_POOL`; `22_features` gives `clean_entries` / `clean_refs`, so the
automatic matcher and the features read the same cleaned columns (`title_clean`, `first_surname`, `year`,
`arxiv_hint`, `doi_norm`, `ref_*`). Both are run with `RUN_TEST = False` (definitions only).

In [ ]:
from pathlib import Path

if "RUN_TEST" not in globals():
    RUN_TEST = True
_RUN_TEST_21 = RUN_TEST
RUN_TEST = False
if "load_m3" not in globals() and Path("./20_m3_data.ipynb").is_file():
    get_ipython().run_line_magic("run", "-i ./20_m3_data.ipynb")
if "clean_entries" not in globals():
    get_ipython().run_line_magic("run", "-i ./22_features.ipynb")   # loads 13 itself when 20 is not there
RUN_TEST = _RUN_TEST_21

import json
import random
import re
import time
import unittest

import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process

SEED = 42
# From 20 when it is loaded; the values of the plan otherwise (offline tests).
M3_PARSER_VERSION = globals().get("M3_PARSER_VERSION", "13.8|10:team|11:team|12:team")
MANUAL_POOL = globals().get("MANUAL_POOL", {"min_entries": 20, "max_entries": 60, "min_refs": 10})
SPLIT_POOL = globals().get("SPLIT_POOL", {"min_entries": 20, "min_refs": 10})
if "member_of" not in globals():
    member_of = lambda arxiv_id: None   # noqa: E731  (20 not loaded: no member ranges)

MANUAL_PER_MEMBER = 5         # statement: at least 5 publications per member
MANUAL_BIBITEM = 2            # of the 5: publications whose references come from \bibitem (the rest from .bib)
MIN_LABELLED_ENTRIES = 20     # statement: at least 20 labelled entries per manual publication
MIN_MANUAL_ENTRIES = 100      # statement: at least 100 manual entries for the group
AUTO_MIN_SHARE = 0.10         # statement: automatic labels on at least 10% of the other publications
AUTO_FRACTION = 0.25          # share of the eligible non-manual publications given automatic labels (see 4.)

MANUAL_DIR = LABEL_DIR / "manual"
TODO_DIR = LABEL_DIR / "todo"
SELECTION_PATH = LABEL_DIR / "selection.csv"
AUTO_LABELS_PATH = LABEL_DIR / "auto" / "auto_labels.parquet"
SPLIT_PATH = LABEL_DIR / "split.json"
LABELS_PATH = LABEL_DIR / "labels.parquet"
LABEL_REPORT_PATH = LOG_DIR / "m3_labels.json"

SHEET_READ_COLUMNS = ["position", "citation_key", "entry_title", "entry_authors", "entry_year", "entry_venue",
                      "hint_1", "hint_2", "hint_3"]
SHEET_FILL_COLUMNS = ["ref_arxiv_id", "confidence", "note", "labeler"]
SHEET_COLUMNS = SHEET_READ_COLUMNS + SHEET_FILL_COLUMNS + ["arxiv_id", "parser_version"]
REFS_SHEET_COLUMNS = ["ref_arxiv_id", "ref_title", "ref_authors", "ref_year", "ref_venue", "link"]
LABEL_OUT_COLUMNS = ["arxiv_id", "citation_key", "ref_arxiv_id", "source", "labeler", "confidence", "rule", "score"]
print(f"21 loaded | labels: {LABEL_DIR} | parser {M3_PARSER_VERSION}")

## 1. Publications to label by hand

`manual_candidates(pubs, member)` lists the publications of a member's range that fit the plan:
20–60 entries, at least 10 arXiv references, M2 and S2 both `ok` (`MANUAL_POOL`). `pick_manual` takes 5 per
member with a fixed seed, `MANUAL_BIBITEM` of them from `\bibitem` lists and the rest from `.bib` files, so
both kinds of BibTeX entries are in the ground truth. The other candidates stay listed in
`labels/todo/<member>/candidates.csv`: a member who finds a publication unusable (another language, a
thesis with 60 identical entries) swaps it there **before** labelling starts and tells the group.

In [ ]:
def _pubs_frame(pubs):
    p = pubs.copy()
    p["arxiv_id"] = p["arxiv_id"].astype(str)
    for column, default in (("m2_status", "ok"), ("s2_status", "ok"), ("bib_source", "unknown")):
        if column not in p.columns:
            p[column] = default
    if "member" not in p.columns:
        p["member"] = [member_of(a) for a in p["arxiv_id"]]
    if "eligible" not in p.columns:
        p["eligible"] = (p["n_entries"] >= 1) & (p["n_refs"] >= 1)
    return p


def manual_candidates(pubs, member, min_entries=None, max_entries=None, min_refs=None):
    """Publications of `member` that fit the manual pool, in ID order."""
    pool = {**MANUAL_POOL, **{k: v for k, v in (("min_entries", min_entries), ("max_entries", max_entries),
                                                 ("min_refs", min_refs)) if v is not None}}
    p = _pubs_frame(pubs)
    mask = ((p["member"] == member) & p["eligible"].astype(bool) & (p["m2_status"] == "ok") & (p["s2_status"] == "ok")
            & p["n_entries"].between(pool["min_entries"], pool["max_entries"]) & (p["n_refs"] >= pool["min_refs"]))
    columns = ["arxiv_id", "member", "bib_source", "n_entries", "n_refs"]
    return p.loc[mask, columns].sort_values("arxiv_id").reset_index(drop=True)


def pick_manual(pubs, members=None, per_member=MANUAL_PER_MEMBER, n_bibitem=MANUAL_BIBITEM, seed=SEED):
    """{member: candidates DataFrame with a `picked` column}: per_member picked, n_bibitem of them from \\bibitem."""
    if members is None:
        members = sorted(MEMBER_RANGES) if "MEMBER_RANGES" in globals() else sorted(set(_pubs_frame(pubs)["member"].dropna()))
    out = {}
    for member in members:
        cand = manual_candidates(pubs, member)
        rng = random.Random(f"{seed}-{member}")
        bibitem = sorted(cand.loc[cand["bib_source"] == "bibitem", "arxiv_id"])
        bib = sorted(cand.loc[cand["bib_source"] != "bibitem", "arxiv_id"])
        take_item = min(n_bibitem, len(bibitem))
        picked = rng.sample(bibitem, take_item)
        picked += rng.sample(bib, min(per_member - take_item, len(bib)))
        rest = sorted(set(bibitem) - set(picked))
        picked += rng.sample(rest, min(per_member - len(picked), len(rest)))   # too few .bib papers
        if len(picked) < per_member:
            print(f"[21] {member}: only {len(picked)} publications fit the manual pool {MANUAL_POOL}")
        cand["picked"] = cand["arxiv_id"].isin(picked)
        out[member] = cand
    return out


def write_selection(picks, path=None, overwrite=False):
    """labels/selection.csv from pick_manual(); never overwritten once labelling may have started."""
    path = Path(path or SELECTION_PATH)
    if path.is_file() and not overwrite:
        print(f"[21] {path} exists: kept (overwrite=True only before anyone starts labelling)")
        return read_selection(path)
    rows = pd.concat([c[c["picked"]] for c in picks.values()], ignore_index=True)
    rows = rows[["member", "arxiv_id", "bib_source", "n_entries", "n_refs"]]
    path.parent.mkdir(parents=True, exist_ok=True)
    rows.to_csv(path, index=False)
    print(f"[21] wrote {path}: {len(rows)} publications, {rows.groupby('member').size().to_dict()}")
    return rows


def read_selection(path=None):
    path = Path(path or SELECTION_PATH)
    if not path.is_file():
        return pd.DataFrame(columns=["member", "arxiv_id", "bib_source", "n_entries", "n_refs"])
    return pd.read_csv(path, dtype={"arxiv_id": str, "member": str})

## 2. Label sheets

`export_label_sheet(arxiv_id, entries, refs)` writes two CSV files to open in Google Sheets
(*File → Import → Upload*, "Replace spreadsheet", **no** conversion of text to numbers):

- **`<fid>.csv`**, one row per entry in bibliography order. To read: `position`, `citation_key`,
  `entry_title`, `entry_authors` (3 at most), `entry_year`, `entry_venue` (journal, else URL), and `hint_1..3`:
  the 3 references whose title is closest (`fuzz.token_set_ratio`), written `ref_arxiv_id | title`. The hints
  only help to find the reference fast; they are **not** labels. To fill: `ref_arxiv_id` (empty = no
  match; `h1`, `h2`, `h3` = that hint), `confidence` (`sure` / `unsure`), `note`, `labeler`.
  `arxiv_id` and `parser_version` are filled already.
- **`<fid>_refs.csv`**: every arXiv reference of the publication (`ref_arxiv_id, ref_title, ref_authors,
  ref_year, ref_venue`) sorted by title, with its `https://arxiv.org/abs/<id>` link.

The sheet never shows the automatic label, so the manual labels stay an independent measure of the
automatic matcher.

In [ ]:
def _authors_short(authors, n=3):
    names = [str(a) for a in (list(authors) if authors is not None and not isinstance(authors, str) else
                              [authors] if authors else [])]
    return "; ".join(names[:n]) + (" et al." if len(names) > n else "")


def _hint_texts(e, r):
    """Title used for the hints: the cleaned title, else the raw text (a \\bibitem entry without a title)."""
    left = [t or raw for t, raw in zip(e["title_clean"], e["raw_text"])]
    right = [t or raw for t, raw in zip(r["ref_title_clean"], r["ref_text"])]
    return left, right


def export_label_sheet(arxiv_id, entries, refs, out_dir=None, n_hints=3):
    """Write <fid>.csv and <fid>_refs.csv of one publication; returns their paths."""
    out_dir = Path(out_dir or TODO_DIR)
    arxiv_id = str(arxiv_id)
    e = entries[entries["arxiv_id"].astype(str) == arxiv_id]
    r = refs[refs["arxiv_id"].astype(str) == arxiv_id]
    if not len(e):
        raise ValueError(f"{arxiv_id} has no entries")
    e = e if "title_clean" in e.columns else clean_entries(e)
    r = r if "ref_title_clean" in r.columns else clean_refs(r)
    e = e.sort_values("position").reset_index(drop=True)
    r = r.reset_index(drop=True)
    hints = [[""] * n_hints for _ in range(len(e))]
    if len(r):
        left, right = _hint_texts(e, r)
        scores = process.cdist(left, right, scorer=fuzz.token_set_ratio, workers=1)
        for i in range(len(e)):
            order = sorted(range(len(r)), key=lambda j: (-scores[i, j], j))[:n_hints]
            hints[i][:len(order)] = [f"{r.at[j, 'ref_arxiv_id']} | {_text(r.at[j, 'ref_title'])[:100]}" for j in order]
    sheet = pd.DataFrame({
        "position": e["position"].astype(int), "citation_key": e["citation_key"],
        "entry_title": [_text(t) for t in e["title"]],
        "entry_authors": [_authors_short(a) for a in e["authors"]],
        "entry_year": [("" if y is pd.NA else str(y)) for y in e["year"]],
        "entry_venue": [_text(j) or _text(u) for j, u in zip(e["journal"], e["url"])],
        **{f"hint_{k + 1}": [h[k] for h in hints] for k in range(n_hints)},
        "ref_arxiv_id": "", "confidence": "", "note": "", "labeler": "",
        "arxiv_id": arxiv_id, "parser_version": M3_PARSER_VERSION})
    refs_sheet = pd.DataFrame({
        "ref_arxiv_id": r["ref_arxiv_id"], "ref_title": [_text(t) for t in r["ref_title"]],
        "ref_authors": [_authors_short(a) for a in r["ref_authors"]],
        "ref_year": [("" if y is pd.NA else str(y)) for y in r["ref_year"]],
        "ref_venue": [_text(v) for v in r["ref_venue"]],
        "link": [f"https://arxiv.org/abs/{x}" for x in r["ref_arxiv_id"]]}).sort_values(
            "ref_title", key=lambda s: s.str.lower())
    out_dir.mkdir(parents=True, exist_ok=True)
    fid = to_folder_id(arxiv_id)
    paths = out_dir / f"{fid}.csv", out_dir / f"{fid}_refs.csv"
    sheet.to_csv(paths[0], index=False, encoding="utf-8")
    refs_sheet.to_csv(paths[1], index=False, encoding="utf-8")
    return paths


def export_sheets(selection, entries, refs, out_dir=None):
    """Sheets of every selected publication, into out_dir/<member>/."""
    out_dir = Path(out_dir or TODO_DIR)
    papers = set(selection["arxiv_id"].astype(str))
    e = clean_entries(entries[entries["arxiv_id"].astype(str).isin(papers)])
    r = clean_refs(refs[refs["arxiv_id"].astype(str).isin(papers)])
    written = []
    for member, arxiv_id in selection[["member", "arxiv_id"]].itertuples(index=False):
        written.append(export_label_sheet(arxiv_id, e, r, out_dir / str(member))[0])
    print(f"[21] {len(written)} label sheets in {out_dir}")
    return written

## 3. Reading and checking the manual labels

A spreadsheet changes some values on the way back: `2504.10000` becomes the number `2504.1`, a citation key
such as `1-2` becomes a date. `read_manual` repairs what it can and says so:

- `repair_arxiv_id`: `h1`/`h2`/`h3` (or `1`/`2`/`3`) → the id of that hint; `arXiv:2504.06381v2`, an arxiv.org
  URL → `2504.06381`; a number with lost trailing zeros → padded again (5 digits after the dot from
  `1501`, 4 before), the form `normalize_ref_arxiv_id` of `03` gives;
- `arxiv_id` comes from the file name (`2605-07338.csv`, also `2605-07338 - Sheet1.csv`);
- a `citation_key` that is not in `entries` is recovered from `position` when the title at that position is
  the same (`norm_title`), otherwise it is an error.

`validate_labels()` returns `{"ok", "errors", "warnings", "labels", "papers", "totals"}`. Errors: missing
columns; a publication not in `entries`; an entry missing, repeated or unknown (the parser changed); a
`ref_arxiv_id` that is not a reference of the publication; another `parser_version`; fewer than 20 labelled
entries. Warnings: a reference given to two entries, an empty or unknown `confidence`, an empty `labeler`,
a member with fewer than 5 publications, fewer than 100 entries in all.

In [ ]:
_FID_RE = re.compile(r"(\d{4}-\d{4,5})")
_HINT_RE = re.compile(r"^h?([1-9])$", re.I)
_NUMERIC_ID_RE = re.compile(r"^(\d{4})\.(\d{1,5})$")
_ARXIV_URL_RE = re.compile(r"^(?:https?://)?(?:www\.|export\.)?arxiv\.org/(?:abs|pdf)/", re.I)


def repair_arxiv_id(value, hints=None):
    """Label cell -> canonical arXiv id, None (no match), or the cleaned text when it is not an id."""
    s = _text(value).strip().strip("'\"")
    if not s or s.lower() in {"nan", "none", "no", "-", "0"}:
        return None
    m = _HINT_RE.match(s)
    if m and hints is not None:
        k = int(m.group(1)) - 1
        hint = _text(hints[k]) if k < len(hints) else ""
        return repair_arxiv_id(hint.split("|")[0]) if hint else s
    s = _ARXIV_URL_RE.sub("", s)
    s = re.sub(r"\.pdf$", "", s, flags=re.I)
    m = _NUMERIC_ID_RE.match(s)
    if m:
        width = 5 if int(m.group(1)) >= 1501 else 4
        s = f"{m.group(1)}.{m.group(2).ljust(width, '0')}"
    return _arxiv_id(s) or s


def _paper_of_file(path, frame):
    m = _FID_RE.search(Path(path).name)
    if m:
        return m.group(1).replace("-", ".")
    ids = {repair_arxiv_id(x) for x in frame.get("arxiv_id", [])} - {None}
    return ids.pop() if len(ids) == 1 else None


def read_manual(manual_dir=None, entries=None, selection=None):
    """(labels, problems) of every sheet in manual_dir; problems: [(level, file, message)]."""
    manual_dir = Path(manual_dir or MANUAL_DIR)
    problems, frames = [], []
    selection = read_selection() if selection is None else selection
    member_by_paper = dict(zip(selection["arxiv_id"].astype(str), selection["member"].astype(str)))
    keys_by_paper, title_at = {}, {}
    if entries is not None:
        e = entries.copy()
        if "position" not in e.columns:   # bibliography order = row order (as clean_entries of 22)
            e["position"] = e.groupby("arxiv_id", sort=False).cumcount()
        e = e[["arxiv_id", "citation_key", "position", "title"]]
        e["arxiv_id"] = e["arxiv_id"].astype(str)
        for a, g in e.groupby("arxiv_id"):
            keys_by_paper[a] = set(g["citation_key"])
            title_at[a] = {int(p): (k, H12["norm_title"](_text(t))) for k, p, t in g[["citation_key", "position", "title"]].itertuples(index=False)}
    for path in sorted(manual_dir.glob("*.csv")):
        if path.name.endswith("_refs.csv") or path.name == SELECTION_PATH.name:
            continue
        df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
        df.columns = [c.strip() for c in df.columns]
        missing = {"citation_key", "ref_arxiv_id"} - set(df.columns)
        if missing:
            problems.append(("error", path.name, f"missing columns {sorted(missing)}"))
            continue
        arxiv_id = _paper_of_file(path, df)
        if arxiv_id is None:
            problems.append(("error", path.name, "no publication id in the file name or the arxiv_id column"))
            continue
        df = df.loc[[any(_text(v).strip() for v in row) for row in df.itertuples(index=False)]]   # blank rows
        hints = df[[c for c in ("hint_1", "hint_2", "hint_3") if c in df.columns]].values.tolist() \
            if "hint_1" in df.columns else [None] * len(df)
        keys = keys_by_paper.get(arxiv_id)
        rows = []
        for (i, row), hint in zip(df.iterrows(), hints):
            key = _text(row["citation_key"]).strip()
            if keys is not None and key not in keys:
                pos = pd.to_numeric(row.get("position", ""), errors="coerce")
                found = title_at.get(arxiv_id, {}).get(int(pos)) if pd.notna(pos) else None
                if found and found[1] == H12["norm_title"](_text(row.get("entry_title", ""))):
                    problems.append(("warning", path.name, f"row {i + 2}: key {key!r} read as {found[0]!r} "
                                     "(same position and title; the spreadsheet changed it)"))
                    key = found[0]
            rows.append({
                "arxiv_id": arxiv_id, "citation_key": key,
                "ref_arxiv_id": repair_arxiv_id(row["ref_arxiv_id"], hint),
                "raw_ref": _text(row["ref_arxiv_id"]).strip(),
                "confidence": _text(row.get("confidence", "")).strip().lower(),
                "note": _text(row.get("note", "")).strip(),
                "labeler": _text(row.get("labeler", "")).strip() or member_by_paper.get(arxiv_id, ""),
                "parser_version": _text(row.get("parser_version", "")).strip(),
                "file": path.name})
        frames.append(pd.DataFrame(rows))
    columns = ["arxiv_id", "citation_key", "ref_arxiv_id", "raw_ref", "confidence", "note", "labeler",
               "parser_version", "file"]
    labels = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns)
    labels["ref_arxiv_id"] = labels["ref_arxiv_id"].astype(object).where(labels["ref_arxiv_id"].notna(), None)
    return labels[columns], problems


def validate_labels(manual_dir=None, entries=None, refs=None, selection=None, parser_version=None):
    """Check every manual sheet against entries/refs; see the markdown above for the rules."""
    parser_version = parser_version or M3_PARSER_VERSION
    if entries is None:
        entries = load_m3("entries", columns=["arxiv_id", "citation_key", "position", "title"])
    if refs is None:
        refs = load_m3("refs", columns=["arxiv_id", "ref_arxiv_id"])
    selection = read_selection() if selection is None else selection
    labels, problems = read_manual(manual_dir, entries, selection)
    errors = [f"{f}: {m}" for level, f, m in problems if level == "error"]
    warnings_ = [f"{f}: {m}" for level, f, m in problems if level == "warning"]
    e_keys = entries.assign(arxiv_id=entries["arxiv_id"].astype(str)).groupby("arxiv_id")["citation_key"].apply(set)
    r_ids = refs.assign(arxiv_id=refs["arxiv_id"].astype(str)).groupby("arxiv_id")["ref_arxiv_id"].apply(set)
    papers = []
    for (arxiv_id, file), g in labels.groupby(["arxiv_id", "file"], sort=True):
        where = f"{file} ({arxiv_id})"
        if arxiv_id not in e_keys.index:
            errors.append(f"{where}: not a publication of entries.parquet")
            continue
        keys, refs_here = e_keys[arxiv_id], r_ids.get(arxiv_id, set())
        dup = g["citation_key"][g["citation_key"].duplicated()].unique().tolist()
        unknown = sorted(set(g["citation_key"]) - keys)
        missing = sorted(keys - set(g["citation_key"]))
        if dup:
            errors.append(f"{where}: entries labelled twice {dup[:5]}")
        if unknown:
            errors.append(f"{where}: {len(unknown)} citation keys not in entries (parser changed?) {unknown[:5]}")
        if missing:
            errors.append(f"{where}: {len(missing)} entries without a row {missing[:5]}")
        given = g[g["ref_arxiv_id"].notna()]
        foreign = given[~given["ref_arxiv_id"].isin(refs_here)]
        for key, raw in foreign[["citation_key", "raw_ref"]].itertuples(index=False):
            errors.append(f"{where}: {key}: {raw!r} is not an arXiv reference of this publication")
        versions = set(g["parser_version"]) - {""}
        if versions - {parser_version}:
            errors.append(f"{where}: parser_version {sorted(versions)}, labels must be made on {parser_version}")
        if g["citation_key"].nunique() < MIN_LABELLED_ENTRIES:
            errors.append(f"{where}: {g['citation_key'].nunique()} labelled entries, at least {MIN_LABELLED_ENTRIES}")
        twice = given["ref_arxiv_id"][given["ref_arxiv_id"].duplicated()].unique().tolist()
        if twice:
            warnings_.append(f"{where}: references given to two entries {twice[:5]} (one reference, one entry)")
        bad_conf = int((~g["confidence"].isin(["sure", "unsure"])).sum())
        if bad_conf:
            warnings_.append(f"{where}: {bad_conf} rows with an empty or unknown confidence (read as sure)")
        if (g["labeler"] == "").any():
            warnings_.append(f"{where}: {int((g['labeler'] == '').sum())} rows without labeler")
        papers.append({"arxiv_id": arxiv_id, "file": file, "labeler": g["labeler"].mode().iat[0] if len(g) else "",
                       "member": member_of(arxiv_id), "entries": len(g), "matched": int(g["ref_arxiv_id"].notna().sum()),
                       "unsure": int((g["confidence"] == "unsure").sum()), "n_refs": len(refs_here)})
    papers = pd.DataFrame(papers, columns=["arxiv_id", "file", "labeler", "member", "entries", "matched", "unsure", "n_refs"])
    if papers["arxiv_id"].duplicated().any():
        errors.append(f"publications with two sheets: {papers.loc[papers['arxiv_id'].duplicated(), 'arxiv_id'].tolist()}")
    per_member = papers.groupby(papers["member"].fillna("none")).size().to_dict()
    for member in (sorted(MEMBER_RANGES) if "MEMBER_RANGES" in globals() else []):
        if per_member.get(member, 0) < MANUAL_PER_MEMBER:
            warnings_.append(f"{member}: {per_member.get(member, 0)} manual publications, the statement asks {MANUAL_PER_MEMBER}")
    if papers["entries"].sum() < MIN_MANUAL_ENTRIES:
        warnings_.append(f"{int(papers['entries'].sum())} manual entries in all, the statement asks {MIN_MANUAL_ENTRIES}")
    labels = labels.assign(confidence=labels["confidence"].where(labels["confidence"].isin(["sure", "unsure"]), "sure"))
    total = int(papers["entries"].sum())
    totals = {"papers": len(papers), "entries": total, "matched": int(papers["matched"].sum()),
              "no_match_share": round(float(1 - papers["matched"].sum() / total), 4) if total else None,
              "unsure_share": round(float(papers["unsure"].sum() / total), 4) if total else None,
              "papers_per_member": {str(k): int(v) for k, v in per_member.items()}}
    return {"ok": not errors, "errors": errors, "warnings": warnings_, "labels": labels, "papers": papers,
            "totals": totals}


def print_validation(result, limit=40):
    print(f"[21] validate_labels: {'OK' if result['ok'] else 'ERRORS'} | {result['totals']}")
    for title, items in (("errors", result["errors"]), ("warnings", result["warnings"])):
        if items:
            print(f"  {title} ({len(items)}):")
            for item in items[:limit]:
                print("   -", item)

## 4. Automatic matcher

For every entry of a publication, the rules are tried in order and the first one that fires gives the
label (high precision first, as in the plan):

| Rule | Fires when | `score` |
|---|---|---|
| `arxiv` | the arXiv id the entry gives (`arxiv_hint`: `url`, `journal`, a `10.48550/arXiv.*` DOI) is a reference id | 100 |
| `doi` | the entry DOI equals the reference DOI (both `norm_doi`) | 100 |
| `title_exact` | the cleaned titles are equal, the title has at least `min_tokens` words, and only one reference has it | 100 |
| `title_fuzzy` | `fuzz.token_sort_ratio` of the cleaned titles ≥ `fuzzy`, the best reference is not tied, the title has at least `min_tokens` words, **and** the first-author surname is equal or the years differ by at most `year_tol` | the ratio |
| `none` | nothing fired | 0 |

Then **one reference, one entry**: when two entries get the same reference, the stronger rule (then the
higher score, then the earlier entry) keeps it and the other becomes `none`. `token_sort_ratio` and not
`token_set_ratio`: the set version gives 100 to a short title contained in a long one.

`auto_label_all` applies the matcher to a seeded sample of `AUTO_FRACTION` (25%) of the eligible publications
outside the manual pool: 2.5 times the 10% the statement asks, while keeping the pairs of `22` small enough for
a Colab CPU session (`fraction=1.0` labels them all). `tune_auto_params` chooses `fuzzy` and `min_tokens` on the
**train** manual publications only (largest recall with a precision of at least 95%), and `auto_vs_manual`
reports precision, recall and the confusion table on all manual publications (poster).

In [ ]:
AUTO_PARAMS = {"fuzzy": 90, "min_tokens": 4, "year_tol": 1}
AUTO_RULES = ("arxiv", "doi", "title_exact", "title_fuzzy")
AUTO_GRID = [{"fuzzy": f, "min_tokens": t, "year_tol": 1} for f in (80, 85, 90, 95) for t in (3, 4, 5)]
AUTO_MIN_PRECISION = 0.95


def _ensure_clean(entries, refs, papers=None):
    """Cleaned (entries, refs) of `papers` (all when None); frames already cleaned are only filtered."""
    if papers is not None:
        papers = set(map(str, papers))
        entries = entries[entries["arxiv_id"].astype(str).isin(papers)]
        refs = refs[refs["arxiv_id"].astype(str).isin(papers)]
    e = entries if "title_clean" in entries.columns else clean_entries(entries)
    r = refs if "ref_title_clean" in refs.columns else clean_refs(refs)
    return e, r


def auto_match_pub(e, r, params=None):
    """Automatic labels of one publication: one row per entry (arxiv_id, citation_key, ref_arxiv_id, rule, score)."""
    p = {**AUTO_PARAMS, **(params or {})}
    e = e.reset_index(drop=True)
    r = r.reset_index(drop=True)
    ref_ids = r["ref_arxiv_id"].tolist()
    by_id = {x: j for j, x in enumerate(ref_ids)}
    dois = r["ref_doi_norm"].tolist()
    by_doi = {d: j for j, d in enumerate(dois) if d and dois.count(d) == 1}
    titles = r["ref_title_clean"].tolist()
    by_title = {}
    for j, t in enumerate(titles):
        by_title.setdefault(t, []).append(j)
    sim = (process.cdist(e["title_clean"].tolist(), titles, scorer=fuzz.token_sort_ratio, workers=1)
           if len(e) and len(r) else np.zeros((len(e), len(r))))
    ref_years = r["ref_year"].tolist()
    ref_surnames = r["ref_first_surname"].tolist()
    proposals = []   # (entry index, ref index, rule rank, score)
    for i, row in enumerate(e[["arxiv_hint", "doi_norm", "title_clean", "first_surname", "year"]].itertuples(index=False)):
        hint, doi, title, surname, year = row
        n_tokens = len(title.split())
        if hint and hint in by_id:
            proposals.append((i, by_id[hint], 0, 100.0))
        elif doi and doi in by_doi:
            proposals.append((i, by_doi[doi], 1, 100.0))
        elif n_tokens >= p["min_tokens"] and len(by_title.get(title, ())) == 1:
            proposals.append((i, by_title[title][0], 2, 100.0))
        elif n_tokens >= p["min_tokens"] and len(r):
            row_sim = sim[i]
            j = int(np.argmax(row_sim))
            best = float(row_sim[j])
            tied = (row_sim >= best).sum() > 1
            same_author = bool(surname) and surname == ref_surnames[j]
            close_year = (year is not pd.NA and ref_years[j] is not pd.NA and abs(int(year) - int(ref_years[j])) <= p["year_tol"])
            if best >= p["fuzzy"] and not tied and (same_author or close_year):
                proposals.append((i, j, 3, best))
    winner = {}
    for i, j, rank, score in sorted(proposals, key=lambda x: (x[2], -x[3], x[0])):
        winner.setdefault(j, (i, rank, score))   # one reference, one entry
    label = {i: (j, rank, score) for j, (i, rank, score) in winner.items()}
    rows = []
    for i, key in enumerate(e["citation_key"]):
        if i in label:
            j, rank, score = label[i]
            rows.append((e.at[i, "arxiv_id"], key, ref_ids[j], AUTO_RULES[rank], round(score, 2)))
        else:
            rows.append((e.at[i, "arxiv_id"], key, None, "none", 0.0))
    return pd.DataFrame(rows, columns=["arxiv_id", "citation_key", "ref_arxiv_id", "rule", "score"])


def auto_match(entries_c, refs_c, papers, params=None):
    """auto_match_pub over `papers` (cleaned frames)."""
    papers = sorted(set(map(str, papers)))
    e_groups = {a: g for a, g in entries_c.groupby(entries_c["arxiv_id"].astype(str), sort=False)}
    r_groups = {a: g for a, g in refs_c.groupby(refs_c["arxiv_id"].astype(str), sort=False)}
    empty_r = refs_c.iloc[:0]
    frames = [auto_match_pub(e_groups[a], r_groups.get(a, empty_r), params) for a in papers if a in e_groups]
    if not frames:
        return pd.DataFrame(columns=["arxiv_id", "citation_key", "ref_arxiv_id", "rule", "score"])
    return pd.concat(frames, ignore_index=True)


def auto_label_all(entries, refs, pubs, exclude=(), params=None, fraction=AUTO_FRACTION, seed=SEED, save=True,
                   path=None):
    """Automatic labels of `fraction` of the eligible publications outside `exclude` (every entry of each one)."""
    t0 = time.time()
    p = _pubs_frame(pubs)
    pool = sorted(set(p.loc[p["eligible"].astype(bool), "arxiv_id"]) - set(map(str, exclude)))
    n = len(pool) if fraction >= 1 else max(1, round(fraction * len(pool)))
    papers = sorted(random.Random(seed).sample(pool, min(n, len(pool))))
    share = len(papers) / len(pool) if pool else 0.0
    if share < AUTO_MIN_SHARE:
        raise ValueError(f"automatic labels on {share:.1%} of the publications, the statement asks {AUTO_MIN_SHARE:.0%}")
    e, r = _ensure_clean(entries, refs, papers)
    auto = auto_match(e, r, papers, params)
    if save:
        path = Path(path or AUTO_LABELS_PATH)
        path.parent.mkdir(parents=True, exist_ok=True)
        auto.to_parquet(path, index=False)
    counts = auto["rule"].value_counts().to_dict()
    print(f"[21] auto_label_all: {len(papers):,} of {len(pool):,} eligible publications outside the manual pool "
          f"({share:.1%}), {len(auto):,} entries, rules {counts} | {time.time() - t0:.0f}s")
    return auto


def _outcome(gold, pred):
    if gold is None and pred is None:
        return "both_none"
    if pred is None:
        return "missed"
    if gold is None:
        return "false_match"
    return "correct" if gold == pred else "wrong_ref"


def auto_vs_manual(manual, entries, refs, params=None):
    """Automatic matcher on the manual publications vs their manual labels: overall, by_rule, confusion, rows."""
    papers = sorted(set(manual["arxiv_id"].astype(str)))
    e, r = _ensure_clean(entries, refs, papers)
    auto = auto_match(e, r, papers, params).rename(columns={"ref_arxiv_id": "auto_ref"})
    rows = manual[["arxiv_id", "citation_key", "ref_arxiv_id"]].merge(auto, on=["arxiv_id", "citation_key"], how="left")
    rows["rule"] = rows["rule"].fillna("none")
    gold = [None if pd.isna(x) else x for x in rows["ref_arxiv_id"]]
    pred = [None if pd.isna(x) else x for x in rows["auto_ref"]]
    rows["outcome"] = [_outcome(g, q) for g, q in zip(gold, pred)]
    c = rows["outcome"].value_counts().reindex(["correct", "wrong_ref", "missed", "false_match", "both_none"], fill_value=0)
    predicted = c["correct"] + c["wrong_ref"] + c["false_match"]
    actual = c["correct"] + c["wrong_ref"] + c["missed"]
    precision = float(c["correct"] / predicted) if predicted else None
    recall = float(c["correct"] / actual) if actual else None
    f1 = 2 * precision * recall / (precision + recall) if precision and recall else 0.0
    by_rule = (rows[rows["rule"] != "none"].groupby("rule")["outcome"]
               .agg(n="size", correct=lambda s: int((s == "correct").sum())).reset_index())
    by_rule["precision"] = (by_rule["correct"] / by_rule["n"]).round(4)
    overall = {"papers": len(papers), "entries": len(rows), "precision": None if precision is None else round(precision, 4),
               "recall": None if recall is None else round(recall, 4), "f1": round(f1, 4),
               "entry_accuracy": round(float((c["correct"] + c["both_none"]) / len(rows)), 4) if len(rows) else None}
    return {"overall": overall, "by_rule": by_rule, "confusion": {k: int(v) for k, v in c.items()}, "rows": rows}


def tune_auto_params(manual_train, entries, refs, grid=AUTO_GRID, min_precision=AUTO_MIN_PRECISION):
    """(params, table): the largest recall with precision >= min_precision on the train manual publications."""
    papers = sorted(set(manual_train["arxiv_id"].astype(str)))
    e, r = _ensure_clean(entries, refs, papers)
    table = []
    for params in grid:
        o = auto_vs_manual(manual_train, e, r, params)["overall"]
        table.append({**params, **{k: o[k] for k in ("precision", "recall", "f1")}})
    table = pd.DataFrame(table)
    ok = table[table["precision"].fillna(0) >= min_precision]
    pick = (ok if len(ok) else table).sort_values(["recall", "precision", "fuzzy", "min_tokens"],
                                                  ascending=[False, False, False, False]).iloc[0]
    if not len(ok):
        print(f"[21] no automatic matcher setting reaches precision {min_precision}: tell the group (plan, section 6)")
    params = {k: int(pick[k]) for k in ("fuzzy", "min_tokens", "year_tol")}
    return params, table

## 5. Split

`pick_split(counts, seed)` draws 2 publications (test, then valid) at random with a fixed seed from those with
at least 20 labelled entries and at least 10 arXiv references (`SPLIT_POOL`), sorted by ID so the draw does
not depend on the order of the files. The manual pair is drawn **first**, before the automatic matcher is
tuned, so its thresholds never see the test and valid publications; the automatic pair is drawn after
`auto_label_all`. Every other labelled publication is train.

In [ ]:
def split_pool(labels, refs, min_entries=None, min_refs=None):
    """Publications of `labels` with enough labelled entries and arXiv references, in ID order."""
    min_entries = SPLIT_POOL["min_entries"] if min_entries is None else min_entries
    min_refs = SPLIT_POOL["min_refs"] if min_refs is None else min_refs
    n_entries = labels.groupby(labels["arxiv_id"].astype(str))["citation_key"].nunique()
    n_refs = refs.groupby(refs["arxiv_id"].astype(str)).size()
    ok = [a for a, n in n_entries.items() if n >= min_entries and n_refs.get(a, 0) >= min_refs]
    return sorted(ok)


def pick_split(pool, seed=SEED, salt=""):
    """{"test": id, "valid": id} drawn from `pool`."""
    if len(pool) < 2:
        raise ValueError(f"the split needs 2 publications, the pool has {len(pool)}")
    test, valid = random.Random(f"{seed}{salt}").sample(sorted(pool), 2)
    return {"test": test, "valid": valid}


def make_split(manual_pick, auto_pick, seed=SEED, path=None, save=True):
    """split.json from the manual and automatic picks."""
    split = {"seed": seed,
             "test": {"manual": manual_pick["test"], "auto": auto_pick["test"]},
             "valid": {"manual": manual_pick["valid"], "auto": auto_pick["valid"]}}
    if save:
        path = Path(path or SPLIT_PATH)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(split, indent=1), encoding="utf-8")
    return split

## 6. Labels for `22_features`

`build_labels(manual, auto)` puts both kinds together with `source` = `manual` / `auto`; a publication that
has manual labels keeps only those. `load_labels()` reads `labels/labels.parquet` (written by `run_labeling`);
when 21 is loaded, `run_features()` of 22 calls it instead of reading the files itself.

In [ ]:
def build_labels(manual, auto=None):
    """Manual + automatic labels in LABEL_OUT_COLUMNS; manual publications keep only their manual labels."""
    m = manual.assign(source="manual", rule="manual", score=np.nan)
    frames = [m]
    if auto is not None and len(auto):
        a = auto[~auto["arxiv_id"].astype(str).isin(set(m["arxiv_id"].astype(str)))]
        frames.append(a.assign(source="auto", labeler="auto", confidence="sure"))
    out = pd.concat(frames, ignore_index=True)
    out["arxiv_id"] = out["arxiv_id"].astype(str)
    out["ref_arxiv_id"] = out["ref_arxiv_id"].astype(object).where(out["ref_arxiv_id"].notna(), None)
    return out[LABEL_OUT_COLUMNS].reset_index(drop=True)


def load_labels(path=None):
    """labels/labels.parquet as a DataFrame (arxiv_id, citation_key, ref_arxiv_id, source, ...)."""
    path = Path(path or LABELS_PATH)
    if not path.is_file():
        raise FileNotFoundError(f"{path} does not exist: run run_labeling() of 21 first")
    labels = pd.read_parquet(path)
    labels["ref_arxiv_id"] = labels["ref_arxiv_id"].astype(object).where(labels["ref_arxiv_id"].notna(), None)
    return labels

## 7. Full run

Two moments:

1. **Before labelling** (once, on Colab with Drive mounted): `prepare_manual_sheets()` picks the 20
   publications, writes `labels/selection.csv` (never overwritten) and the sheets in `labels/todo/<member>/`,
   and prints the numbers of each member's pool.
2. **After labelling** (`25_run_m3`): `run_labeling()` checks the manual sheets (stops on an error), draws the
   manual test/valid pair, tunes the matcher on the train manual publications, labels the automatic sample,
   draws the automatic pair, writes `auto_labels.parquet`, `split.json`, `labels.parquet` and
   `logs/m3_labels.json` (with precision and recall against all manual labels).

In [ ]:
def prepare_manual_sheets(entries=None, refs=None, pubs=None, seed=SEED, out_dir=None, overwrite=False):
    """Pick the manual publications, write selection.csv and the label sheets; returns the selection."""
    pubs = load_m3("pubs") if pubs is None else pubs
    picks = pick_manual(pubs, seed=seed)
    selection = write_selection(picks, overwrite=overwrite)
    entries = load_m3("entries", columns=list(BIBTEX_COLUMNS) + ["bib_source", "position"]) if entries is None else entries
    refs = load_m3("refs") if refs is None else refs
    out_dir = Path(out_dir or TODO_DIR)
    export_sheets(selection, entries, refs, out_dir)
    for member, cand in picks.items():
        (out_dir / member).mkdir(parents=True, exist_ok=True)
        cand.to_csv(out_dir / member / "candidates.csv", index=False)
        print(f"[21] {member}: pool {len(cand)} ({int((cand['bib_source'] == 'bibitem').sum())} bibitem), "
              f"picked {cand.loc[cand['picked'], 'arxiv_id'].tolist()}")
    return selection


def run_labeling(entries=None, refs=None, pubs=None, params=None, tune=True, fraction=AUTO_FRACTION, seed=SEED,
                 manual_dir=None, save=True):
    """Manual check -> split (manual) -> tuning -> automatic labels -> split (auto) -> files. Returns a dict."""
    t0 = time.time()
    pubs = load_m3("pubs") if pubs is None else pubs
    entries = load_m3("entries", columns=list(BIBTEX_COLUMNS) + ["bib_source", "position"]) if entries is None else entries
    refs = load_m3("refs") if refs is None else refs
    check = validate_labels(manual_dir, entries, refs)
    print_validation(check)
    if not check["ok"]:
        raise ValueError("fix the manual sheets first (errors above)")
    manual = check["labels"]
    manual_ids = sorted(set(manual["arxiv_id"]))
    manual_pick = pick_split(split_pool(manual, refs), seed, salt="manual")
    held_out = set(manual_pick.values())
    manual_train = manual[~manual["arxiv_id"].isin(held_out)]
    e_m, r_m = _ensure_clean(entries, refs, manual_ids)
    tuning = None
    if tune:
        params, tuning = tune_auto_params(manual_train, e_m, r_m)
        print(f"[21] automatic matcher parameters (train manual publications): {params}")
    params = {**AUTO_PARAMS, **(params or {})}
    auto = auto_label_all(entries, refs, pubs, exclude=manual_ids, params=params, fraction=fraction, seed=seed,
                          save=save)
    auto_pick = pick_split(split_pool(auto, refs), seed, salt="auto")
    split = make_split(manual_pick, auto_pick, seed, save=save)
    report = auto_vs_manual(manual, e_m, r_m, params)
    labels = build_labels(manual, auto)
    summary = {
        "manual": check["totals"], "manual_papers": check["papers"].to_dict("records"),
        "auto": {"papers": int(auto["arxiv_id"].nunique()), "entries": len(auto),
                 "matched": int(auto["ref_arxiv_id"].notna().sum()), "rules": auto["rule"].value_counts().to_dict(),
                 "fraction": fraction},
        "params": params, "tuning": None if tuning is None else tuning.to_dict("records"),
        "auto_vs_manual": {**report["overall"], "confusion": report["confusion"],
                           "by_rule": report["by_rule"].to_dict("records")},
        "split": split, "seconds": round(time.time() - t0, 1)}
    if save:
        LABELS_PATH.parent.mkdir(parents=True, exist_ok=True)
        labels.to_parquet(LABELS_PATH, index=False)
        LABEL_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
        LABEL_REPORT_PATH.write_text(json.dumps(summary, indent=1, default=str), encoding="utf-8")
        print(f"[21] wrote {AUTO_LABELS_PATH}, {SPLIT_PATH}, {LABELS_PATH}, {LABEL_REPORT_PATH}")
    print(f"[21] auto vs manual: {report['overall']} | {report['confusion']}")
    print(f"[21] split: {split} | {time.time() - t0:.0f}s")
    return {"labels": labels, "auto": auto, "split": split, "check": check, "report": report, "summary": summary}

## 8. Tests

Offline, on publications built in memory (`_toy`), cleaned by `22`. They cover the rules of the matcher
(new and old arXiv ids, DOI, exact and fuzzy titles, the author/year guard, short titles, one reference
per entry), the sheets, the repair of spreadsheet values, `validate_labels`, the split and `build_labels`.
Then a demonstration on the local dev sample when it exists (`dev_frames` of `22`).

In [ ]:
def _toy_entry(paper, key, title, authors=("Ann Smith",), year=2020, url=None, journal=None, doi=None, pos=0,
               bib_source="bib"):
    return {"arxiv_id": paper, "citation_key": key, "entry_type": "article", "title": title, "authors": list(authors),
            "year": year, "doi": doi, "journal": journal, "url": url, "bib_source": bib_source, "position": pos}


def _toy_ref(paper, ref_id, title, authors=("Ann Smith",), year=2020, doi=None, pos=0):
    return {"arxiv_id": paper, "ref_arxiv_id": ref_id, "ref_title": title, "ref_authors": list(authors),
            "ref_submission_date": f"{year}-01-01", "ref_s2_id": f"s{pos}", "ref_year": year, "ref_doi": doi,
            "ref_venue": None, "ref_position": pos}


def _toy():
    P = "2605.07338"
    entries = pd.DataFrame([
        _toy_entry(P, "new", "Some paper", url="https://arxiv.org/abs/2504.06381v2", pos=0),
        _toy_entry(P, "old", "Old physics", journal="arXiv:hep-th/9901001", pos=1),
        _toy_entry(P, "doi", "Published version", doi="10.1103/PhysRevD.1", pos=2),
        _toy_entry(P, "exact", "Learning to rank with gradient boosted trees", pos=3),
        _toy_entry(P, "fuzzy", "Deep residual learning for image recognition", authors=("Kaiming He",), year=2016, pos=4),
        _toy_entry(P, "guard", "Graph attention networks for molecules", authors=("Zed Zulu",), year=2010, pos=5),
        _toy_entry(P, "short", "Adam", pos=6),
        _toy_entry(P, "book", "The art of computer programming", authors=("Donald Knuth",), year=1968, pos=7),
    ])
    refs = pd.DataFrame([
        _toy_ref(P, "2504.06381", "A different title", pos=0),
        _toy_ref(P, "hep-th/9901001", "Old physics paper", pos=1),
        _toy_ref(P, "2101.00002", "Preprint version", doi="10.1103/physrevd.1", pos=2),
        _toy_ref(P, "2101.00003", "Learning to Rank with Gradient-Boosted Trees", pos=3),
        _toy_ref(P, "1512.03385", "Deep residual learning for image recognition (extended)", authors=("Kaiming He",),
                 year=2015, pos=4),
        _toy_ref(P, "2101.00005", "Graph attention networks for molecule", authors=("Petar V",), year=2018, pos=5),
        _toy_ref(P, "1412.6980", "Adam", pos=6),
    ])
    return entries, refs


class LabelTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.entries, cls.refs = _toy()
        cls.e, cls.r = clean_entries(cls.entries), clean_refs(cls.refs)
        cls.auto = auto_match_pub(cls.e, cls.r).set_index("citation_key")

    # ---------- automatic matcher ----------
    def test_every_entry_gets_a_row(self):
        self.assertEqual(sorted(self.auto.index), sorted(self.entries["citation_key"]))

    def test_arxiv_rule_new_and_old_ids(self):
        self.assertEqual(tuple(self.auto.loc["new", ["ref_arxiv_id", "rule"]]), ("2504.06381", "arxiv"))
        self.assertEqual(tuple(self.auto.loc["old", ["ref_arxiv_id", "rule"]]), ("hep-th/9901001", "arxiv"))

    def test_doi_rule(self):
        self.assertEqual(tuple(self.auto.loc["doi", ["ref_arxiv_id", "rule"]]), ("2101.00002", "doi"))

    def test_exact_title_after_cleaning(self):
        self.assertEqual(tuple(self.auto.loc["exact", ["ref_arxiv_id", "rule"]]), ("2101.00003", "title_exact"))

    def test_fuzzy_title_with_same_author(self):
        e, r = self.e.copy(), self.r.copy()
        out = auto_match_pub(e, r, {"fuzzy": 80}).set_index("citation_key")
        self.assertEqual(tuple(out.loc["fuzzy", ["ref_arxiv_id", "rule"]]), ("1512.03385", "title_fuzzy"))

    def test_fuzzy_needs_author_or_year(self):
        out = auto_match_pub(self.e, self.r, {"fuzzy": 80}).set_index("citation_key")
        self.assertEqual(out.loc["guard", "rule"], "none")   # similar title, other author, 8 years apart

    def test_short_title_and_book_are_not_matched(self):
        self.assertEqual(self.auto.loc["short", "rule"], "none")   # "adam": fewer than min_tokens words
        self.assertEqual(self.auto.loc["book", "rule"], "none")

    def test_one_reference_one_entry(self):
        P = "2605.07338"
        e = clean_entries(pd.DataFrame([
            _toy_entry(P, "a", "Attention is all you need today", pos=0),
            _toy_entry(P, "b", "Attention is all you need", url="arxiv.org/abs/1706.03762", pos=1)]))
        r = clean_refs(pd.DataFrame([_toy_ref(P, "1706.03762", "Attention is all you need", pos=0)]))
        out = auto_match_pub(e, r, {"fuzzy": 80}).set_index("citation_key")
        self.assertEqual(out.loc["b", "rule"], "arxiv")
        self.assertEqual(out.loc["a", "rule"], "none")

    def test_auto_vs_manual_counts(self):
        manual = pd.DataFrame({"arxiv_id": "2605.07338", "citation_key": ["new", "doi", "exact", "short", "book"],
                               "ref_arxiv_id": ["2504.06381", "2101.00009", "2101.00003", "1412.6980", None]})
        rep = auto_vs_manual(manual, self.e, self.r)
        self.assertEqual(rep["confusion"], {"correct": 2, "wrong_ref": 1, "missed": 1, "false_match": 0, "both_none": 1})
        self.assertAlmostEqual(rep["overall"]["precision"], 2 / 3, places=3)
        self.assertAlmostEqual(rep["overall"]["recall"], 2 / 4, places=3)

    # ---------- sheets and manual labels ----------
    def test_export_label_sheet(self):
        import tempfile
        with tempfile.TemporaryDirectory() as tmp:
            sheet, refs_sheet = export_label_sheet("2605.07338", self.e, self.r, tmp)
            s = pd.read_csv(sheet, dtype=str, keep_default_na=False)
            rs = pd.read_csv(refs_sheet, dtype=str, keep_default_na=False)
        self.assertEqual(len(s), len(self.entries))
        self.assertEqual(list(s.columns), SHEET_COLUMNS)
        self.assertTrue(s.loc[s["citation_key"] == "exact", "hint_1"].iat[0].startswith("2101.00003 |"))
        self.assertTrue((s["ref_arxiv_id"] == "").all())
        self.assertEqual(len(rs), len(self.refs))

    def test_repair_arxiv_id(self):
        self.assertEqual(repair_arxiv_id("2504.1"), "2504.10000")
        self.assertEqual(repair_arxiv_id("1412.698"), "1412.6980")
        self.assertEqual(repair_arxiv_id("arXiv:2504.06381v2"), "2504.06381")
        self.assertEqual(repair_arxiv_id("https://arxiv.org/abs/hep-th/9901001"), "hep-th/9901001")
        self.assertEqual(repair_arxiv_id("h2", ["x | a", "2101.00003 | b", ""]), "2101.00003")
        self.assertIsNone(repair_arxiv_id("  "))

    def _write_sheets(self, tmp, edit=None):
        """Sheet of the toy paper, filled with correct labels, then edited by `edit(df)`."""
        sheet, _ = export_label_sheet("2605.07338", self.e, self.r, tmp)
        s = pd.read_csv(sheet, dtype=str, keep_default_na=False)
        gold = {"new": "2504.06381", "old": "hep-th/9901001", "doi": "2101.00002", "exact": "h1"}
        s["ref_arxiv_id"] = s["citation_key"].map(gold).fillna("")
        s["confidence"], s["labeler"] = "sure", "TV3"
        if edit:
            s = edit(s)
        s.to_csv(sheet, index=False)
        return Path(tmp)

    def test_validate_labels_ok_and_hint(self):
        import tempfile
        with tempfile.TemporaryDirectory() as tmp:
            res = validate_labels(self._write_sheets(tmp), self.entries, self.refs, selection=read_selection(Path(tmp) / "x"))
        self.assertEqual([e for e in res["errors"] if "at least" not in e], [])
        lab = res["labels"].set_index("citation_key")
        self.assertEqual(lab.loc["exact", "ref_arxiv_id"], "2101.00003")
        self.assertIsNone(lab.loc["book", "ref_arxiv_id"])

    def test_validate_labels_catches_errors(self):
        import tempfile

        def edit(s):
            s = pd.concat([s, s.iloc[[0]]])                      # entry labelled twice
            s = s[s["citation_key"] != "book"]                   # entry missing
            s.loc[s["citation_key"] == "short", "ref_arxiv_id"] = "2399.99999"   # not a reference of the paper
            s.loc[s["citation_key"] == "old", ["citation_key", "entry_title"]] = ["renamed", "Other"]   # key not in entries
            s["parser_version"] = "13.7|10:team|11:team|12:team"                  # other parser
            return s
        with tempfile.TemporaryDirectory() as tmp:
            res = validate_labels(self._write_sheets(tmp, edit), self.entries, self.refs,
                                  selection=read_selection(Path(tmp) / "x"))
        text = "\n".join(res["errors"])
        self.assertFalse(res["ok"])
        for needle in ("labelled twice", "without a row", "2399.99999", "not in entries", "parser_version"):
            self.assertIn(needle, text)

    def test_key_changed_by_spreadsheet_is_recovered(self):
        import tempfile

        def edit(s):
            s.loc[s["citation_key"] == "fuzzy", "citation_key"] = "2-Jan"   # what Sheets made of the key
            return s
        with tempfile.TemporaryDirectory() as tmp:
            res = validate_labels(self._write_sheets(tmp, edit), self.entries, self.refs,
                                  selection=read_selection(Path(tmp) / "x"))
        self.assertNotIn("not in entries", "\n".join(res["errors"]))
        self.assertIn("fuzzy", set(res["labels"]["citation_key"]))
        self.assertTrue(any("spreadsheet changed it" in w for w in res["warnings"]))

    # ---------- split and labels ----------
    def test_split_is_reproducible_and_uses_the_pool(self):
        labels = pd.DataFrame({"arxiv_id": [f"2605.0{i:04d}" for i in range(6) for _ in range(25)],
                               "citation_key": [f"k{j}" for _ in range(6) for j in range(25)]})
        refs = pd.DataFrame({"arxiv_id": [f"2605.0{i:04d}" for i in range(6) for _ in range(12 if i else 5)],
                             "ref_arxiv_id": "x"})
        pool = split_pool(labels, refs)
        self.assertNotIn("2605.00000", pool)                 # only 5 references
        a, b = pick_split(pool, 42, "manual"), pick_split(list(reversed(pool)), 42, "manual")
        self.assertEqual(a, b)
        self.assertNotEqual(a["test"], a["valid"])

    def test_build_labels_manual_wins(self):
        manual = pd.DataFrame({"arxiv_id": ["p1"], "citation_key": ["a"], "ref_arxiv_id": [None], "labeler": ["TV3"],
                               "confidence": ["sure"]})
        auto = pd.DataFrame({"arxiv_id": ["p1", "p2"], "citation_key": ["a", "a"], "ref_arxiv_id": ["x", "y"],
                             "rule": ["arxiv", "doi"], "score": [100.0, 100.0]})
        out = build_labels(manual, auto)
        self.assertEqual(out[["arxiv_id", "source", "ref_arxiv_id"]].values.tolist(),
                         [["p1", "manual", None], ["p2", "auto", "y"]])
        self.assertEqual(list(out.columns), LABEL_OUT_COLUMNS)


if RUN_TEST:
    _res = unittest.TextTestRunner(verbosity=1).run(unittest.defaultTestLoader.loadTestsFromTestCase(LabelTests))
    assert _res.wasSuccessful(), "21 tests failed"

In [ ]:
# Dev demo: the matcher and one sheet on the local dev sample (no manual labels there).
if RUN_TEST and "DEV_BIBTEX" in globals() and DEV_BIBTEX.is_file():
    _t0 = time.time()
    dev_entries, dev_refs = dev_frames()
    dev_e, dev_r = clean_entries(dev_entries), clean_refs(dev_refs)
    dev_auto = auto_match(dev_e, dev_r, sorted(dev_e["arxiv_id"].unique()))
    print(f"dev: {dev_auto['arxiv_id'].nunique()} papers, {len(dev_auto)} entries, rules "
          f"{dev_auto['rule'].value_counts().to_dict()} | {time.time() - _t0:.1f}s")
    _busy = dev_r["arxiv_id"].value_counts().index[0]
    _paths = export_label_sheet(_busy, dev_e, dev_r, LOG_DIR / "m3_dev" / "labels" / "todo")
    display(pd.read_csv(_paths[0], dtype=str, keep_default_na=False).head(5))